# Stage 2 · Inference variants on trained weights (no training)

This notebook is the protocol for comparing **inference settings** on already trained detectors. It takes explicit checkpoints,
runs the same tiled full-image inference with each variant, evaluates dev and val with `auric.evaluation`, and compares variants
with a **paired image-bootstrap** (`auric.evaluation.compare`). All logic lives in `src/auric`; the notebook re-implements none of it.

**Why it exists** (`docs/04_research_decision_report.md`, finding C2). `model.predict` emits only the best class of each box, while
Ultralytics' own validator emits every class above the threshold (`multi_label=True`). Where two classes are plausible for one object
(Cargo vs Box) the second hypothesis can only be a true positive if it is emitted. The upper-end estimate on a classification proxy
was a third of the mAP; on the real detectors it is smaller and has to be measured.

| Stage | What happens | Written to `OUT_DIR/<model>/<variant>/` |
|---|---|---|
| 0 Setup | runtime, code, explicit inputs (checkpoints, variants, output folder) | |
| 1 Data | class map, annotations, the shipped dev split | |
| 2 Inference + evaluation | each model x variant on dev and val | `predictions/<split>.csv.gz`, `eval/<split>/`, `record.json` |
| 3 Results | headline and per-class tables, paired comparisons | |

**Integrity.** Dev decides, val is reported. A variant is judged by its paired dev difference; val is only shown next to it.
Existing outputs are never overwritten unless `OVERWRITE = True`.

## Stage 0 · Setup

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

RUNTIME = "kaggle"       # "kaggle" | "local": selects configs/env/<RUNTIME>.yaml and where the code is imported from

if RUNTIME == "kaggle":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    SRC_INPUT = Path("/kaggle/input/datasets/kumarshutosh/version-a1-src/src/auric")      # uploaded package (read-only)
    SRC_DIR = Path("/kaggle/working/src")
    shutil.copytree(SRC_INPUT, SRC_DIR / "auric", dirs_exist_ok=True, ignore=shutil.ignore_patterns("__pycache__"))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.179"], check=True)
elif RUNTIME == "local":
    SRC_DIR = Path("C:/Auric/src")
else:
    raise ValueError(f"unknown RUNTIME {RUNTIME!r}")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
ENV_FILE = SRC_DIR / "auric" / "configs" / "env" / f"{RUNTIME}.yaml"       # data_root, work_dir, device, workers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 5.6 MB/s eta 0:00:00


In [3]:
# Explicit inputs. Each model = (experiment config in src/auric/configs, trained checkpoint). The inference section of that
# config (tile, overlap, imgsz, merge_iou, ...) is what the checkpoint was evaluated with; a variant overrides some of it.
if RUNTIME == "local":
    WEIGHTS_DIR = Path("C:/Auric/weights")                       # A1_last.pt, D1_last.pt: the last.pt of each Kaggle run
    OUT_DIR = Path("C:/Auric/outputs/inference_variants")
    REFERENCE_DIR = Path("C:/Auric/outputs")                      # <model>/<model>_predictions/{dev,val}.csv.gz from Kaggle
else:
    A1_WEIGHTS_DIR = Path("/kaggle/input/notebooks/kumarshutosh/auric-versiona1/work/runs/A1_s0/train/weights")
    D1_WEIGHTS_DIR = Path("/kaggle/input/notebooks/shriramprajapati/version-d1/work/runs/D1_s0/train/weights")
    # adjust to the dataset that holds the checkpoints
    OUT_DIR = Path("/kaggle/working/inference_variants")
    REFERENCE_DIR = None
MODELS = {"A1": ("A1", A1_WEIGHTS_DIR / "last.pt"), "D1": ("D1", D1_WEIGHTS_DIR / "last.pt")}

# Variants = overrides of cfg["inference"]. "single" must reproduce the Kaggle evaluation; "multi_label" emits every class
# above conf for a box. Its image cap is raised because it produces several times more low-score boxes per image.
VARIANTS = {"single": {}, "multi_label": {"multi_label": True, "max_det_image": 5000}}

# Run-time decisions (deliberately not config values)
SMOKE = False           # True: 3 images per split and output under OUT_DIR/smoke (wiring check)
OVERWRITE = False       # False: an existing output folder is an error
VERBOSE = False
if SMOKE:
    OUT_DIR = OUT_DIR / "smoke"
missing = [str(w) for _, w in MODELS.values() if not Path(w).is_file()]
if missing:
    raise FileNotFoundError(f"checkpoints not found: {missing}")

## Stage 1 · Data

In [4]:
import numpy as np
import pandas as pd
import torch
from IPython.display import display

from auric.config import load_config
from auric.data.dataset import index_split, load_annotations, read_class_names
from auric.data.splits import ensure_split, subset
from auric.evaluation import compare, evaluate
from auric.model.predict import TiledPredictor
from auric.runs import Run, log_stage, sha256
from auric.structures import load_detections, save_detections

pd.set_option("display.precision", 3)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

CFGS = {m: load_config(SRC_DIR / "auric" / "configs" / f"{exp}.yaml", ENV_FILE) for m, (exp, _) in MODELS.items()}
cfg0 = next(iter(CFGS.values()))
DATA_ROOT, SPLIT_FILE = Path(cfg0["paths"]["data_root"]), Path(cfg0["paths"]["split_file"])
DEVICE = cfg0["runtime"]["device"]
EC = cfg0["evaluation"]
NAMES = read_class_names(DATA_ROOT)
K = len(NAMES)
SHORT = [n.replace("Truck w/", "").replace("Truck ", "").replace(" Truck", "") for n in NAMES]
RECORDS = {s: index_split(DATA_ROOT, s) for s in ("train", "val")}
ANNS = {s: load_annotations(RECORDS[s], K) for s in RECORDS}
split, _ = ensure_split(SPLIT_FILE, ANNS["train"], NAMES, **cfg0["split"])
EVAL_SETS = {"dev": (subset(RECORDS["train"], split["dev"]), ANNS["train"]), "val": (RECORDS["val"], ANNS["val"])}
if SMOKE:
    EVAL_SETS = {p: (recs[:3], anns) for p, (recs, anns) in EVAL_SETS.items()}
GTS = {p: {r.image_id: anns[r.image_id] for r in recs} for p, (recs, anns) in EVAL_SETS.items()}
print(f"torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'} | device={DEVICE}")
print({p: f"{len(g)} images, {sum(len(a) for a in g.values())} objects" for p, g in GTS.items()})

torch 2.11.0+cu128 | GPU: Tesla T4 | device=0
{'dev': '66 images, 1093 objects', 'val': '22 images, 1552 objects'}


## Stage 2 · Inference and evaluation

For every model and variant: tiled inference on dev and val (`TiledPredictor`), the detections are saved, and the evaluation
(`evaluate`, COCO 101-point mAP50, 1000-resample image bootstrap) is saved next to them with a `record.json` holding the checkpoint
hash, the effective inference settings and the environment.

In [5]:
DETS, REPORTS = {}, {}
for model, (exp, weights) in MODELS.items():
    weights_sha = sha256(weights)
    for vname, override in VARIANTS.items():
        ic = {**CFGS[model]["inference"], **override}
        run = Run(OUT_DIR / model / vname)
        if run.dir.exists() and any(run.dir.iterdir()):
            if not OVERWRITE:
                raise FileExistsError(f"{run.dir} exists; set OVERWRITE = True to replace it")
            shutil.rmtree(run.dir)
        predictor = TiledPredictor(weights, ic, DEVICE)
        for part, (recs, _) in EVAL_SETS.items():
            dets = predictor.predict(recs, desc=f"{model}/{vname} {part}")
            save_detections(dets, run.predictions(part))
            rep = evaluate(GTS[part], dets, NAMES, EC["size_bins"], EC["bootstrap"], EC["seed"])
            rep.summary["settings"] = {"evaluation": EC, "inference": ic, "weights_sha256": weights_sha}
            rep.save(run.eval_dir(part))
            log_stage(run, f"evaluate_{part}", {"weights": str(weights), "weights_sha256": weights_sha, "config": exp,
                                                "inference": ic, "evaluation": EC, "mAP50": rep.summary["mAP50"],
                                                "images": [r.image_id for r in recs]})
            DETS[(model, vname, part)], REPORTS[(model, vname, part)] = dets, rep
# the Kaggle evaluation of the same checkpoints, if its prediction files are available (reproduction check)
if REFERENCE_DIR is not None and not SMOKE:
    for model in MODELS:
        for part in EVAL_SETS:
            f = REFERENCE_DIR / model / f"{model}_predictions" / f"{part}.csv.gz"
            if f.is_file():
                DETS[(model, "kaggle", part)] = load_detections(f)
                REPORTS[(model, "kaggle", part)] = evaluate(GTS[part], DETS[(model, "kaggle", part)], NAMES,
                                                             EC["size_bins"], EC["bootstrap"], EC["seed"])
print(len(REPORTS), "evaluations")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


A1/single dev:   0%|          | 0/66 [00:00<?, ?image/s]

A1/single val:   0%|          | 0/22 [00:00<?, ?image/s]

A1/multi_label dev:   0%|          | 0/66 [00:00<?, ?image/s]

A1/multi_label val:   0%|          | 0/22 [00:00<?, ?image/s]

D1/single dev:   0%|          | 0/66 [00:00<?, ?image/s]

D1/single val:   0%|          | 0/22 [00:00<?, ?image/s]

D1/multi_label dev:   0%|          | 0/66 [00:00<?, ?image/s]

D1/multi_label val:   0%|          | 0/22 [00:00<?, ?image/s]

8 evaluations


## Stage 3 · Results

In [6]:
def headline(r):
    s, op = r.summary, r.summary["operating_point"]
    ci = s["mAP50_ci95"] or [np.nan, np.nan]
    return {"mAP50": s["mAP50"], "CI95": f"[{ci[0]:.3f}, {ci[1]:.3f}]", "AP50 class-agnostic": s["AP50_class_agnostic"],
            "mAP50-95": s["mAP50_95"], "mAP75": s["mAP75"], "n_det": s["n_det"], "conf*": op["conf"],
            "P@conf*": op["precision"], "R@conf*": op["recall"], "F1@conf*": op["f1"]}


rows = {(m, v, p): headline(r) for (m, v, p), r in REPORTS.items()}
table = pd.DataFrame(rows).T
table.index.names = ["model", "variant", "split"]
display(table.sort_index(level=["split", "model"]))
cap = {k: float(np.mean([len(d) >= VARIANTS.get(k[1], {}).get("max_det_image", CFGS[k[0]]["inference"]["max_det_image"])
                         for d in DETS[k].values()])) for k in DETS if k[1] != "kaggle"}
print("share of images that hit max_det_image:", {k: round(v, 2) for k, v in cap.items() if v > 0})

mAP50            CI95 AP50 class-agnostic mAP50-95  mAP75  n_det  conf* P@conf* R@conf* F1@conf*
model variant     split                                                                                                  
A1    multi_label dev    0.108  [0.076, 0.161]               0.245    0.057  0.053  76908  0.238   0.227   0.277     0.25
      single      dev      0.1  [0.067, 0.152]               0.252    0.053   0.05  46120  0.234   0.227   0.281    0.251
D1    multi_label dev    0.115  [0.083, 0.162]               0.288    0.057   0.05  68926  0.154   0.288   0.247    0.266
      single      dev    0.101  [0.068, 0.147]               0.299     0.05  0.044  39062  0.127   0.251   0.281    0.265
A1    multi_label val    0.142  [0.093, 0.217]               0.294    0.069  0.065  56947  0.136    0.24    0.35    0.285
      single      val    0.131  [0.082, 0.207]               0.303    0.065  0.061  28828  0.134   0.242   0.339    0.282
D1    multi_label val    0.106  [0.072, 0.156]                0.28    0.046  0.033  54535  0.089   0.213   0.334     0.26
      single      val    0.088  [0.053, 0.137]               0.297    0.038  0.026  25253  0.089   0.214   0.288    0.246

share of images that hit max_det_image: {('A1', 'single', 'dev'): 0.03, ('A1', 'single', 'val'): 0.32, ('A1', 'multi_label', 'dev'): 0.02, ('A1', 'multi_label', 'val'): 0.18, ('D1', 'single', 'dev'): 0.03, ('D1', 'single', 'val'): 0.32, ('D1', 'multi_label', 'dev'): 0.02, ('D1', 'multi_label', 'val'): 0.27}


In [8]:
per_class = pd.DataFrame({k: r.per_class.set_index("class")["AP50"] for k, r in REPORTS.items()}).T
per_class.index.names = ["model", "variant", "split"]
per_class.columns = SHORT
display(per_class.sort_index(level=["split", "model"]))

Cargo    Box  Flatbed  Tractor  Liquid
model variant     split                                        
A1    multi_label dev    0.120  0.298    0.052    0.050   0.021
      single      dev    0.118  0.280    0.035    0.052   0.014
D1    multi_label dev    0.114  0.351    0.073    0.018   0.018
      single      dev    0.111  0.325    0.050    0.018   0.000
A1    multi_label val    0.164  0.270    0.073    0.040   0.161
      single      val    0.148  0.255    0.060    0.036   0.158
D1    multi_label val    0.140  0.177    0.144    0.045   0.022
      single      val    0.110  0.155    0.131    0.043   0.000

### Paired comparisons

`compare(gts, A, B)` gives AP50 of B minus A with a **paired** bootstrap interval (same image resamples for both), so it is far
narrower than the two separate intervals above. First each variant against `single` for the same model, then A1 against D1 for the same
variant. `p_b_better` is the share of resamples in which B beats A.

In [9]:
out = {}
for model in MODELS:
    for vname in VARIANTS:
        if vname == "single":
            continue
        for part in EVAL_SETS:
            out[(model, f"{vname} vs single", part)] = compare(GTS[part], DETS[(model, "single", part)],
                                                              DETS[(model, vname, part)], NAMES, EC["bootstrap"], EC["seed"])
for part in EVAL_SETS:
    for vname in VARIANTS:
        if len(MODELS) == 2:
            a, b = MODELS
            out[(f"{b} vs {a}", vname, part)] = compare(GTS[part], DETS[(a, vname, part)], DETS[(b, vname, part)], NAMES,
                                                       EC["bootstrap"], EC["seed"])
for key, df in out.items():
    print(key)
    display(df)

('A1', 'multi_label vs single', 'dev')


,A,B,diff,lo,hi,p_b_better
metric,,,,,,
Cargo Truck,0.118,0.120,0.001,-1.227e-03,0.004,0.804
Truck w/Box,0.280,0.298,0.018,1.208e-02,0.025,1.000
Truck w/Flatbed,0.035,0.052,0.017,8.368e-03,0.031,1.000
Truck Tractor,0.052,0.050,-0.002,-9.435e-03,0.002,0.196
Truck w/Liquid,0.014,0.021,0.007,4.200e-04,0.020,0.982
mAP50,0.100,0.108,0.008,5.269e-03,0.012,1.000
AP50 class-agnostic,0.252,0.245,-0.007,-8.410e-03,-0.005,0.000


('A1', 'multi_label vs single', 'val')


,A,B,diff,lo,hi,p_b_better
metric,,,,,,
Cargo Truck,0.148,0.164,0.016,0.004,0.028,0.994
Truck w/Box,0.255,0.270,0.015,0.007,0.026,1.000
Truck w/Flatbed,0.060,0.073,0.013,-0.004,0.030,0.946
Truck Tractor,0.036,0.040,0.004,-0.003,0.015,0.692
Truck w/Liquid,0.158,0.161,0.003,-0.012,0.019,0.740
mAP50,0.131,0.142,0.010,0.005,0.015,1.000
AP50 class-agnostic,0.303,0.294,-0.009,-0.012,-0.007,0.000


('D1', 'multi_label vs single', 'dev')


,A,B,diff,lo,hi,p_b_better
metric,,,,,,
Cargo Truck,0.111,0.114,2.646e-03,-0.004,0.010,0.745
Truck w/Box,0.325,0.351,2.594e-02,0.016,0.037,1.000
Truck w/Flatbed,0.050,0.073,2.296e-02,0.014,0.035,1.000
Truck Tractor,0.018,0.018,-2.953e-04,-0.009,0.002,0.388
Truck w/Liquid,0.000,0.018,1.825e-02,0.003,0.069,1.000
mAP50,0.101,0.115,1.390e-02,0.009,0.024,1.000
AP50 class-agnostic,0.299,0.288,-1.158e-02,-0.015,-0.008,0.000


... (6 outputs left out)

('D1 vs A1', 'single', 'val')


,A,B,diff,lo,hi,p_b_better
metric,,,,,,
Cargo Truck,0.148,0.110,-0.038,-0.088,0.006,0.050
Truck w/Box,0.255,0.155,-0.100,-0.166,-0.017,0.002
Truck w/Flatbed,0.060,0.131,0.071,-0.005,0.180,0.958
Truck Tractor,0.036,0.043,0.007,-0.037,0.027,0.528
Truck w/Liquid,0.158,0.000,-0.158,-0.446,-0.008,0.000
mAP50,0.131,0.088,-0.044,-0.108,0.003,0.032
AP50 class-agnostic,0.303,0.297,-0.006,-0.042,0.034,0.389


('D1 vs A1', 'multi_label', 'val')


,A,B,diff,lo,hi,p_b_better
metric,,,,,,
Cargo Truck,0.164,0.140,-0.025,-0.063,0.012,0.116
Truck w/Box,0.270,0.177,-0.092,-0.157,-0.014,0.010
Truck w/Flatbed,0.073,0.144,0.071,-0.013,0.195,0.934
Truck Tractor,0.040,0.045,0.005,-0.039,0.025,0.481
Truck w/Liquid,0.161,0.022,-0.138,-0.414,0.003,0.031
mAP50,0.142,0.106,-0.036,-0.101,0.013,0.078
AP50 class-agnostic,0.294,0.280,-0.014,-0.050,0.027,0.250


## Observations (fill in after the run)

* **Reproduction:** does `single` reproduce the Kaggle evaluation (A1 dev 0.100 / val 0.131, D1 dev 0.101 / val 0.088)? Small differences come from GPU numerics (fp16, T4 vs local).
* **Decision on dev:** paired dev difference of `multi_label` vs `single`, overall and per class; val only as a check.
* **Where it helps:** which classes gain (the proxy predicts Box, Flatbed, Tractor most) and whether the image cap mattered.